# Notebook 03 — Hyperparameter Tuning & Run Comparison

**RHOAI 3.5 | MLflow Nested Runs + Dashboard Comparison**

This notebook shows how to:
1. Organise a hyperparameter search using **nested runs** (parent/child hierarchy)
2. Use `mlflow.search_runs()` to find the best configuration
3. Interpret the RHOAI Dashboard comparison views (parallel coordinates, scatter, box, contour)
4. Cross-experiment run comparison

---

### Why Nested Runs?
A parent run represents the overall search job. Each child run is one configuration.
The RHOAI Dashboard displays child runs under their parent, keeping the experiment view clean.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from mlflow.models import infer_signature
from mlflow.sklearn import SERIALIZATION_FORMAT_CLOUDPICKLE
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings("ignore")

EXPERIMENT_NAME = "rhoai-mlflow-demo"
mlflow.set_experiment(EXPERIMENT_NAME)

# Prepare data (same as notebook 02)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Active experiment : {EXPERIMENT_NAME}")
print(f"Tracking URI      : {mlflow.get_tracking_uri()}")

## Section 1 — Grid Search with Nested Runs

We search over `n_estimators` × `max_depth` × `min_samples_split` and log every combination as a **child run** under one **parent run**.

In the RHOAI Dashboard you can compare all child runs with:
- **Parallel coordinates plot** — see how each hyperparameter correlates with accuracy
- **Scatter plot** — any two metrics plotted against each other
- **Box plot** — distribution of a metric across runs
- **"Show differences only"** toggle — hide params that are identical across all selected runs

In [ ]:
# Hyperparameter grid
PARAM_GRID = {
    "n_estimators"      : [50, 100, 200],
    "max_depth"         : [3, 5, None],
    "min_samples_split" : [2, 5],
}

combinations = list(itertools.product(
    PARAM_GRID["n_estimators"],
    PARAM_GRID["max_depth"],
    PARAM_GRID["min_samples_split"],
))
print(f"Total combinations: {len(combinations)}")

best = {"test_acc": 0, "run_id": None, "params": {}}

# --- Parent run: represents the full grid search job ---
with mlflow.start_run(run_name="rf-grid-search-parent") as parent_run:
    mlflow.set_tags({
        "notebook"    : "03_hyperparameter_tuning",
        "search_type" : "grid",
        "rhoai_build" : "3.5",
    })
    mlflow.log_param("total_combinations", len(combinations))
    mlflow.log_param("param_grid", str(PARAM_GRID))

    parent_run_id = parent_run.info.run_id

    for i, (n_est, depth, min_split) in enumerate(combinations, 1):
        # --- Child run: one configuration ---
        run_name = f"rf-n{n_est}-d{depth}-s{min_split}"
        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            params = {
                "n_estimators"      : n_est,
                "max_depth"         : depth,
                "min_samples_split" : min_split,
                "random_state"      : 42,
            }
            mlflow.log_params(params)
            mlflow.set_tag("notebook", "03_hyperparameter_tuning")

            model = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=depth,
                min_samples_split=min_split,
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X_train_s, y_train)

            train_acc = accuracy_score(y_train, model.predict(X_train_s))
            test_acc  = accuracy_score(y_test,  model.predict(X_test_s))
            f1        = f1_score(y_test, model.predict(X_test_s), average="weighted")
            cv_mean   = cross_val_score(model, X_train_s, y_train, cv=5).mean()

            mlflow.log_metrics({
                "train_accuracy" : train_acc,
                "test_accuracy"  : test_acc,
                "f1_weighted"    : f1,
                "cv5_mean"       : cv_mean,
            })

            if test_acc > best["test_acc"]:
                best["test_acc"] = test_acc
                best["run_id"]   = child_run.info.run_id
                best["params"]   = params
                best["model"]    = model

        print(f"  [{i:2d}/{len(combinations)}] n={n_est:3d} depth={str(depth):4s} split={min_split} "
              f"→ test_acc={test_acc:.4f}")

    # Log best result on the parent
    mlflow.log_metric("best_test_accuracy", best["test_acc"])
    mlflow.log_params({f"best_{k}": v for k, v in best["params"].items()})
    mlflow.set_tag("best_child_run_id", best["run_id"])

print(f"\nBest configuration: {best['params']}")
print(f"Best test accuracy : {best['test_acc']:.4f}")
print(f"Best child run ID  : {best['run_id']}")
print(f"Parent run ID      : {parent_run_id}")

## Dashboard: Comparing Runs

Now go to the RHOAI Dashboard and try the comparison views:

```
Develop & train > Experiments (MLflow) > rhoai-mlflow-demo
```

1. Click into the **rf-grid-search-parent** run to see all child runs
2. Select all child runs → click **Compare**
3. Try the following chart types:
   - **Parallel coordinates** — drag axes to reorder, see which `n_estimators`/`max_depth` clusters at high accuracy
   - **Scatter plot** — `cv5_mean` vs `test_accuracy`
   - **Box plot** — distribution of `f1_weighted`
   - Toggle **"Show differences only"** — hides the `random_state` param that's the same in every run
4. For **cross-experiment comparison**: select runs from different experiments and click Compare

## Section 2 — Retrieve and Reproduce the Best Run

In [ ]:
client = mlflow.tracking.MlflowClient()

# Fetch all child runs from the grid search
child_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'",
    order_by=["metrics.test_accuracy DESC"],
)

print(f"Child runs found: {len(child_runs)}")
print()

cols = ["run_id", "tags.mlflow.runName",
        "params.n_estimators", "params.max_depth", "params.min_samples_split",
        "metrics.test_accuracy", "metrics.cv5_mean"]
available = [c for c in cols if c in child_runs.columns]
print(child_runs[available].head(10).to_string(index=False))

In [ ]:
signature = infer_signature(X_train_s, best["model"].predict(X_train_s))

with mlflow.start_run(run_name="best-rf-model") as best_run:
    mlflow.set_tags({
        "notebook"         : "03_hyperparameter_tuning",
        "selected_from"    : "grid-search",
        "parent_run_id"    : parent_run_id,
        "rhoai_build"      : "3.5",
    })
    mlflow.log_params(best["params"])
    mlflow.log_metric("test_accuracy", best["test_acc"])

    mlflow.sklearn.log_model(
        best["model"],
        name="model",
        serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
        signature=signature,
        input_example=X_test_s[:5],
        registered_model_name=None,
    )

    best_final_run_id = best_run.info.run_id
    print(f"Best model logged.")
    print(f"  Run ID        : {best_final_run_id}")
    print(f"  Test accuracy : {best['test_acc']:.4f}")
    print(f"  Params        : {best['params']}")

## Section 3 — Accuracy vs Depth Visualisation

Replicate locally what the dashboard's scatter/contour plots show.

In [ ]:
if len(child_runs) > 0 and "metrics.test_accuracy" in child_runs.columns:
    plot_df = child_runs[
        ["params.n_estimators", "params.max_depth",
         "params.min_samples_split", "metrics.test_accuracy"]
    ].copy().dropna(subset=["metrics.test_accuracy"])

    # Replace None depth with 0 for plotting
    plot_df["params.max_depth"] = plot_df["params.max_depth"].replace({None: "None"})
    plot_df["params.n_estimators"] = plot_df["params.n_estimators"].astype(int)
    plot_df["metrics.test_accuracy"] = plot_df["metrics.test_accuracy"].astype(float)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Test accuracy by n_estimators
    for split_val in ["2", "5"]:
        sub = plot_df[plot_df["params.min_samples_split"] == split_val]
        axes[0].scatter(
            sub["params.n_estimators"],
            sub["metrics.test_accuracy"],
            label=f"min_samples_split={split_val}",
            s=60,
        )
    axes[0].set_xlabel("n_estimators")
    axes[0].set_ylabel("Test Accuracy")
    axes[0].set_title("Accuracy vs n_estimators (mirroring Dashboard scatter plot)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2: Box plot of accuracy by max_depth
    depths = plot_df["params.max_depth"].unique()
    data_by_depth = [
        plot_df[plot_df["params.max_depth"] == d]["metrics.test_accuracy"].tolist()
        for d in depths
    ]
    axes[1].boxplot(data_by_depth, labels=[str(d) for d in depths])
    axes[1].set_xlabel("max_depth")
    axes[1].set_ylabel("Test Accuracy")
    axes[1].set_title("Accuracy Distribution by max_depth (mirroring Dashboard box plot)")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No child run data available for plotting.")

## Summary

| Feature | Details |
|---------|--------|
| Nested runs | `mlflow.start_run(nested=True)` — parent groups all child configs |
| Search child runs | `filter_string="tags.mlflow.parentRunId = '<id>'"` |
| Dashboard comparison | Parallel coordinates, scatter, box, contour — all available in RHOAI 3.5 |
| "Show differences only" | Hides identical params across selected runs |
| Cross-experiment compare | Select runs from different experiments in the dashboard |

**Next:** `04_genai_tracing.ipynb` — LLM call tracing, sessions, and trace archival.